# Week 2 / Session 3 복습 랩 — 힌트 없음

주제: **포트폴리오 비중 관리 — 분산인가 쏠림인가**

## 규칙
1. 각 문제는 **TODO 셀**에 직접 코드를 작성한다.
2. 바로 아래 **CHECK 셀**을 실행해 `✅ PASS`가 나오면 다음으로 넘어간다.
3. `❌ FAIL`이면 TODO 셀을 다시 고친다. 건너뛰지 말 것.
4. **힌트와 정답은 이 랩에 없다.** session3.ipynb 본 강의 자료를 다시 보지 말고 직접 풀어 보라.
5. 모든 CHECK가 `PASS`여야 제출 가능.

---

**체크 목록** (9문제)
- Part 1 준비: Q1 import, Q2 샘플 CSV 생성
- Part 2 비중 계산: Q3 CSV 로드, Q4 평가금액, Q5 총자산 & 비중
- Part 3 구조 분석: Q6 섹터 groupby, Q7 HHI
- Part 4 AI 연결: Q8 요약 dict, Q9 OpenAI 호출

## 사전 설치

In [ ]:
!pip install pandas openai -q

---
# Part 1 — 준비

## Q1. 라이브러리 import

아래 2개를 **모두** import 하라:
- `pandas` as `pd`
- `OpenAI` (from `openai`)

In [ ]:
# TODO Q1


In [ ]:
# CHECK Q1
try:
    assert 'pd' in dir() and pd.__name__ == 'pandas', 'pandas as pd 누락'
    assert 'OpenAI' in dir(), 'openai OpenAI 누락'
    print('✅ PASS Q1')
except AssertionError as e:
    print(f'❌ FAIL Q1: {e}')

## Q2. 샘플 포트폴리오 CSV 생성

아래 스펙으로 `portfolio.csv`를 만들어라.

**컬럼**: `ticker, name, quantity, avg_price, current_price, sector`

**데이터**

| ticker | name | quantity | avg_price | current_price | sector |
|---|---|---|---|---|---|
| NVDA | NVIDIA | 10 | 800 | 950 | AI/반도체 |
| AMD  | AMD    | 15 | 120 | 140 | AI/반도체 |
| TSM  | TSMC   | 20 | 100 | 110 | 반도체 |
| AAPL | Apple  |  5 | 170 | 190 | IT/기기 |
| MSFT | Microsoft | 5 | 400 | 420 | IT/소프트웨어 |

`to_csv("portfolio.csv", index=False, encoding="utf-8-sig")`로 저장.

In [ ]:
# TODO Q2


In [ ]:
# CHECK Q2
import os
try:
    assert os.path.exists('portfolio.csv'), 'portfolio.csv 파일 없음'
    tmp = pd.read_csv('portfolio.csv')
    assert len(tmp) == 5, f'행 개수 오류 (기대 5, 실제 {len(tmp)})'
    for col in ['ticker','name','quantity','avg_price','current_price','sector']:
        assert col in tmp.columns, f'{col} 컬럼 누락'
    assert 'NVDA' in tmp['ticker'].values, 'NVDA 행 없음'
    print('✅ PASS Q2')
except Exception as e:
    print(f'❌ FAIL Q2: {e}')

---
# Part 2 — 비중 계산

## Q3. CSV 로드

`portfolio.csv`를 읽어 `df`에 담아라.

In [ ]:
# TODO Q3


In [ ]:
# CHECK Q3
try:
    assert isinstance(df, pd.DataFrame), 'df가 DataFrame이 아님'
    assert len(df) == 5, f'행 개수 오류 (기대 5, 실제 {len(df)})'
    assert 'quantity' in df.columns and 'current_price' in df.columns, '필수 컬럼 누락'
    print('✅ PASS Q3')
except AssertionError as e:
    print(f'❌ FAIL Q3: {e}')

## Q4. 평가금액 컬럼 추가

`df`에 `value` 컬럼을 추가하라.

$$value = quantity \times current\_price$$

In [ ]:
# TODO Q4


In [ ]:
# CHECK Q4
try:
    assert 'value' in df.columns, 'value 컬럼 없음'
    nvda_value = df.loc[df['ticker']=='NVDA','value'].iloc[0]
    assert nvda_value == 9500, f'NVDA value 기대 9500, 실제 {nvda_value}'
    msft_value = df.loc[df['ticker']=='MSFT','value'].iloc[0]
    assert msft_value == 2100, f'MSFT value 기대 2100, 실제 {msft_value}'
    print('✅ PASS Q4')
except AssertionError as e:
    print(f'❌ FAIL Q4: {e}')

## Q5. 총 자산 & 비중 계산

- `total_value`: `value` 컬럼의 합
- `df["weight"]`: 각 종목이 전체에서 차지하는 비율 (value / total_value)

비중의 합은 반드시 **1.0**이 되어야 한다.

In [ ]:
# TODO Q5


In [ ]:
# CHECK Q5
try:
    assert 'total_value' in dir(), 'total_value 변수 없음'
    assert total_value == 16850, f'total_value 기대 16850, 실제 {total_value}'
    assert 'weight' in df.columns, 'weight 컬럼 없음'
    assert abs(df['weight'].sum() - 1.0) < 1e-6, f'비중 합 1.0 아님 ({df["weight"].sum()})'
    print('✅ PASS Q5')
except AssertionError as e:
    print(f'❌ FAIL Q5: {e}')

---
# Part 3 — 구조 분석

## Q6. 섹터별 비중 집계 (groupby)

`sector_weights`를 만들어라 — 컬럼 `sector`, `weight`의 DataFrame, `weight` 내림차순 정렬.

In [ ]:
# TODO Q6


In [ ]:
# CHECK Q6
try:
    assert isinstance(sector_weights, pd.DataFrame), 'sector_weights가 DataFrame이 아님'
    assert 'sector' in sector_weights.columns and 'weight' in sector_weights.columns, '컬럼 누락'
    assert len(sector_weights) == 4, f'섹터 개수 기대 4, 실제 {len(sector_weights)}'
    ai_weight = sector_weights.loc[sector_weights['sector']=='AI/반도체','weight'].iloc[0]
    assert abs(ai_weight - (9500+2100)/16850) < 1e-6, f'AI/반도체 비중 오류 ({ai_weight})'
    w = sector_weights['weight'].tolist()
    assert w == sorted(w, reverse=True), '내림차순 정렬 아님'
    print('✅ PASS Q6')
except AssertionError as e:
    print(f'❌ FAIL Q6: {e}')

## Q7. HHI (집중도 지표) 계산

$$HHI = \sum w_i^2$$

`hhi` 변수에 저장하고, 기준에 따라 `status` 문자열도 만들어라.

| HHI | status |
|---|---|
| < 0.15 | `"✅ 양호 (잘 분산됨)"` |
| 0.15 ~ 0.25 | `"⚖️ 주의 (집중됨)"` |
| > 0.25 | `"⚠️ 고위험 (매우 집중됨)"` |

In [ ]:
# TODO Q7


In [ ]:
# CHECK Q7
try:
    assert 'hhi' in dir(), 'hhi 변수 없음'
    expected = (sector_weights['weight']**2).sum()
    assert abs(hhi - expected) < 1e-6, f'HHI 값 오류 (기대 {expected:.4f}, 실제 {hhi:.4f})'
    assert 0 < hhi < 1, f'HHI 범위 이상 ({hhi})'
    assert 'status' in dir() and isinstance(status, str), 'status 문자열 없음'
    assert '고위험' in status, f'status 판정 오류 (hhi={hhi:.3f}, status={status})'
    print(f'✅ PASS Q7 (HHI={hhi:.3f})')
except AssertionError as e:
    print(f'❌ FAIL Q7: {e}')

---
# Part 4 — AI 연결

## Q8. AI 비평용 요약 dict

AI 프롬프트에 꽂아 쓰기 좋은 dict `analysis_summary`를 만들어라.

구조:
```python
{
    "sector_data": [ {"sector": ..., "weight": ...}, ... ],
    "hhi_score": 0.354
}
```

In [ ]:
# TODO Q8


In [ ]:
# CHECK Q8
try:
    assert isinstance(analysis_summary, dict), 'dict가 아님'
    assert 'sector_data' in analysis_summary and 'hhi_score' in analysis_summary, '키 누락'
    sd = analysis_summary['sector_data']
    assert isinstance(sd, list) and len(sd) == 4, 'sector_data 리스트 길이 이상'
    assert all('sector' in r and 'weight' in r for r in sd), 'sector_data 원소 형식 오류'
    assert abs(analysis_summary['hhi_score'] - round(hhi,3)) < 1e-9, 'hhi_score 값 오류'
    print('✅ PASS Q8')
except AssertionError as e:
    print(f'❌ FAIL Q8: {e}')

## Q9. OpenAI 호출 — 퀀트 비평

아래 요구대로 AI 비평을 `ai_critique`에 저장하라.

- 모델: `gpt-4o`
- system: "월스트리트 15년차 헤지펀드 퀀트 애널리스트" 역할, 아래 4줄 포맷만 답하게 지시
  - `📊 현재 구조:` / `🎯 핵심 리스크:` / `💡 리밸런싱 제안:` / `⚠️ 모니터링 포인트:`
- user: `analysis_summary`의 섹터 비중과 HHI를 문자열로 넣어 비평 요청
- `temperature=0.3`, `max_tokens=500`

`OPENAI_API_KEY`는 환경변수(`os.environ`) 또는 Colab `userdata`에서 가져온다.

또한 `client` 변수에 `OpenAI` 인스턴스를 만들어 둘 것.

In [ ]:
# TODO Q9


In [ ]:
# CHECK Q9
try:
    assert 'client' in dir() and client.__class__.__name__ == 'OpenAI', 'client 없음'
    assert isinstance(ai_critique, str), 'ai_critique 문자열 아님'
    assert len(ai_critique) > 50, '응답이 너무 짧음'
    print(f'✅ PASS Q9 (길이 {len(ai_critique)})')
except AssertionError as e:
    print(f'❌ FAIL Q9: {e}')

---
# 최종 점검

아래 셀을 실행해 9문제의 PASS 개수를 확인한다.

In [ ]:
# FINAL SCOREBOARD
import os
checks = {
    'Q1': lambda: 'pd' in dir() and 'OpenAI' in dir(),
    'Q2': lambda: os.path.exists('portfolio.csv'),
    'Q3': lambda: isinstance(df, pd.DataFrame) and len(df)==5,
    'Q4': lambda: 'value' in df.columns and df.loc[df['ticker']=='NVDA','value'].iloc[0]==9500,
    'Q5': lambda: total_value==16850 and abs(df['weight'].sum()-1.0)<1e-6,
    'Q6': lambda: isinstance(sector_weights, pd.DataFrame) and len(sector_weights)==4,
    'Q7': lambda: isinstance(hhi, float) and 0<hhi<1 and isinstance(status, str),
    'Q8': lambda: isinstance(analysis_summary, dict) and 'sector_data' in analysis_summary,
    'Q9': lambda: isinstance(ai_critique, str) and len(ai_critique)>50,
}
passed = 0
for q, fn in checks.items():
    try:
        ok = fn()
    except Exception:
        ok = False
    mark = '✅' if ok else '❌'
    print(f'{mark} {q}')
    passed += 1 if ok else 0
print(f'\n=== {passed}/9 PASS ===')
print('합격' if passed == 9 else '불합격 — FAIL 항목 재시도')